In [3]:
import pandas as pd
import torch
import json
import re
import os
from datasets import Dataset
from transformers import TrainingArguments
from tqdm import tqdm


In [1]:

!pip install bitsandbytes transformers accelerate -q

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

torch.cuda.empty_cache()

model_name = "yandex/YandexGPT-5-Lite-8B-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

print("✅ Модель загружена!")

# Тест


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

✅ Модель загружена!


AttributeError: 

In [2]:
import torch

In [4]:
torch.cuda.empty_cache()


In [6]:
!pip install bitsandbytes -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 29.8 MB/s eta 0:00:00


In [4]:
train_df = pd.read_excel("/content/cases_train2 (4).xlsx")
test_df = pd.read_excel("/content/Test1_2 (2).xlsx")


In [5]:
def generate_critique(project_text, scores, model, tokenizer, max_new_tokens=1024):
    if pd.isna(project_text) or project_text == "":
        return "❌ Нет текста проекта"
    system_prompt = f"""Ты — эксперт по оценке бизнес-проектов.

Вот оценки проекта по 5 критериям (каждый от 1 до 5):
1. Анализ ЦА: {scores['ЦА']}/5
2. Проработка решения: {scores['Проработка']}/5
3. Финансовая модель: {scores['Финансы']}/5
4. Анализ рисков: {scores['Риски']}/5
5. Доказательства: {scores['Доказательства']}/5

Напиши РАЗВЕРНУТУЮ КРИТИКУ проекта.

ВАЖНО:
- Напиши УНИКАЛЬНУЮ критику для этого конкретного проекта
- Объясни, ПОЧЕМУ проект получил такие оценки
- Укажи, что можно улучшить
- Будь КОНКРЕТНЫМ, ссылайся на детали проекта
- НЕ КОПИРУЙ шаблонные фразы!
-Не используй смайлики
-Не используй слэнг
-Используй нейтральный стиль

Формат ответа:
1. АНАЛИЗ ЦА (оценка X/5): [твой уникальный комментарий]
2. ПРОРАБОТКА РЕШЕНИЯ (оценка X/5): [твой уникальный комментарий]
3. ФИНАНСОВАЯ МОДЕЛЬ (оценка X/5): [твой уникальный комментарий]
4. АНАЛИЗ РИСКОВ (оценка X/5): [твой уникальный комментарий]
5. ДОКАЗАТЕЛЬСТВА (оценка X/5): [твой уникальный комментарий]

Итоговая оценка: {sum(scores.values()) / 5:.1f}/5
Общий вывод: [твой уникальный вывод]
"""

    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": f"Вот текст проекта для оценки:\n\n{project_text[:3000]}"
        }
    ]

    try:
        inputs = tokenizer.apply_chat_template(
            messages,
            return_tensors="pt",
            add_generation_prompt=True
        ).to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.8,
                do_sample=True,
                top_p=0.95,
                repetition_penalty=1.1,
            )
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        if "assistant" in response:
            parts = response.split("assistant")
            if len(parts) > 1:
                return parts[-1].strip()
        return response
    except Exception as e:
        return f"❌ Ошибка: {str(e)}"


In [11]:
def generate_critique(project_text, scores, model, tokenizer, max_new_tokens=1024):
    if pd.isna(project_text) or project_text == "":
        return "❌ Нет текста проекта"

    system_prompt = f"""Ты — эксперт по оценке бизнес-проектов.

Вот оценки проекта по 5 критериям (каждый от 1 до 5):
1. Анализ ЦА: {scores['ЦА']}/5
2. Проработка решения: {scores['Проработка']}/5
3. Финансовая модель: {scores['Финансы']}/5
4. Анализ рисков: {scores['Риски']}/5
5. Доказательства: {scores['Доказательства']}/5

Напиши РАЗВЕРНУТУЮ КРИТИКУ проекта.

ВАЖНО:
- Напиши УНИКАЛЬНУЮ критику для этого конкретного проекта
- Объясни, ПОЧЕМУ проект получил такие оценки
- Укажи, что можно улучшить
- Будь КОНКРЕТНЫМ, ссылайся на детали проекта
- НЕ КОПИРУЙ шаблонные фразы!
- Не используй смайлики
- Не используй слэнг
- Используй нейтральный стиль

Формат ответа:
1. Анализ ЦА (оценка X/5): [твой уникальный комментарий]
2. Проработка решения (оценка X/5): [твой уникальный комментарий]
3. Финансовая модель (оценка X/5): [твой уникальный комментарий]
4. Анализ рисков (оценка X/5): [твой уникальный комментарий]
5. Доказательства (оценка X/5): [твой уникальный комментарий]

Итоговая оценка: {sum(scores.values()) / 5:.1f}/5
Общий вывод: [твой уникальный вывод]
"""

    # ✅ ПРАВИЛЬНЫЙ ФОРМАТ ДЛЯ YANDEXGPT
    prompt = f"""<|system|>
{system_prompt}
<|user|>
Вот текст проекта для оценки:

{project_text[:3000]}
<|assistant|>
"""

    # Токенизируем
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=True).to("cuda")

    try:
        with torch.no_grad():
            outputs = model.generate(
                inputs.input_ids,
                max_new_tokens=max_new_tokens,
                temperature=0.8,
                do_sample=True,
                top_p=0.95,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id,
            )


        response = tokenizer.decode(outputs[0], skip_special_tokens=True)


        if "<|assistant|>" in response:
            parts = response.split("<|assistant|>")
            if len(parts) > 1:
                return parts[-1].strip()

        return response

    except Exception as e:
        return f"❌ Ошибка: {str(e)}"

In [12]:
test_df["Критика_модели"] = None


In [13]:
for idx in tqdm(test_df.index, desc="Обработка кейсов"):
    project_text = test_df.loc[idx, "Решение кейса"]
    scores = {
        'ЦА': int(test_df.loc[idx, 'ЦА']) if 'ЦА' in test_df.columns else 3,
        'Проработка': int(test_df.loc[idx, 'Проработка решения']) if 'Проработка решения' in test_df.columns else 3,
        'Финансы': int(test_df.loc[idx, 'Финансовая модель и метрики']) if 'Финансовая модель и метрики' in test_df.columns else 3,
        'Риски': int(test_df.loc[idx, 'Анализ рисков']) if 'Анализ рисков' in test_df.columns else 3,
        'Доказательства': int(test_df.loc[idx, 'Доказательства']) if 'Доказательства' in test_df.columns else 3
    }
    critique = generate_critique(project_text, scores, model, tokenizer)
    test_df.loc[idx, "Критика_модели"] = critique
    if (idx + 1) % 5 == 0:
        test_df.to_excel("critique_partial.xlsx", index=False)
        print(f"\n💾 Сохранено {idx + 1} кейсов")


Обработка кейсов:   4%|▍         | 5/130 [02:26<1:01:55, 29.72s/it]


💾 Сохранено 5 кейсов


Обработка кейсов:   8%|▊         | 10/130 [05:03<1:02:21, 31.18s/it]


💾 Сохранено 10 кейсов


Обработка кейсов:  12%|█▏        | 15/130 [07:43<59:13, 30.90s/it]  

...


💾 Сохранено 60 кейсов


Обработка кейсов:  50%|█████     | 65/130 [32:45<30:16, 27.95s/it]


💾 Сохранено 65 кейсов


Обработка кейсов:  50%|█████     | 65/130 [32:50<32:50, 30.31s/it]


KeyboardInterrupt: 

In [ ]:
output_file = "test_with_critique.xlsx"
test_df.to_excel(output_file, index=False)